In [29]:
import pandas as pd
import mlflow


In [30]:
from pathlib import Path

DATA_PATH = Path("./Data")
RAW_DATA_PATH = DATA_PATH / "raw" / "Fraud_Data.csv"
PROCESSED_DATA_PATH = DATA_PATH / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [31]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Fraud Detection")
mlflow.start_run(run_name="DataSplitting")

<ActiveRun: >

# Load Data with correct Types

In [32]:
df = pd.read_csv(
    PROCESSED_DATA_PATH / "2_feature_encoding_result.csv",
    parse_dates=["signup_time", "purchase_time"]
)

In [33]:
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,...,browser_FireFox,browser_IE,browser_Opera,browser_Safari,sex_F,sex_M,source_fr_enc,browser_fr_enc,source_targ_enc,browser_targ_enc
0,286057,2015-01-01 00:00:42,2015-03-25 11:33:06,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
1,309557,2015-01-01 00:00:43,2015-01-01 00:00:44,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
2,124539,2015-01-01 00:00:44,2015-01-01 00:00:45,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
3,161246,2015-01-01 00:00:45,2015-01-01 00:00:46,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
4,356414,2015-01-01 00:00:46,2015-01-01 00:00:47,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 32 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   user_id                       151112 non-null  int64         
 1   signup_time                   151112 non-null  datetime64[ns]
 2   purchase_time                 151112 non-null  datetime64[ns]
 3   purchase_value                151112 non-null  int64         
 4   device_id                     151112 non-null  object        
 5   source                        151112 non-null  object        
 6   browser                       151112 non-null  object        
 7   sex                           151112 non-null  object        
 8   age                           151112 non-null  int64         
 9   ip_address                    151112 non-null  object        
 10  class                         151112 non-null  int64         
 11  time_velocity

# Time Based Splitting

In [35]:
# Sort chronologically
df = df.sort_values("signup_time")

# 80% oldest data -> train
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

In [36]:
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,...,browser_FireFox,browser_IE,browser_Opera,browser_Safari,sex_F,sex_M,source_fr_enc,browser_fr_enc,source_targ_enc,browser_targ_enc
0,286057,2015-01-01 00:00:42,2015-03-25 11:33:06,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
1,309557,2015-01-01 00:00:43,2015-01-01 00:00:44,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
2,124539,2015-01-01 00:00:44,2015-01-01 00:00:45,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
3,161246,2015-01-01 00:00:45,2015-01-01 00:00:46,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
4,356414,2015-01-01 00:00:46,2015-01-01 00:00:47,14,BBPACGBUVJUXF,Ads,Chrome,F,38,119.75.87.223,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798


In [37]:
drop_cols = [
    "user_id",
    "signup_time",
    "purchase_time",
    "device_id",
    "ip_address",
    "class"
]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["class"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["class"]

# Save CSV

In [38]:
X_train.to_csv(PROCESSED_DATA_PATH / "4.1_X_train.csv", index=False)
y_train.to_csv(PROCESSED_DATA_PATH / "4.2_y_train.csv", index=False)
X_test.to_csv(PROCESSED_DATA_PATH / "4.3_X_test.csv", index=False)
y_test.to_csv(PROCESSED_DATA_PATH / "4.4_y_test.csv", index=False)

In [39]:
train_df.to_csv(PROCESSED_DATA_PATH / "4.5_train_df.csv", index=False)
test_df.to_csv(PROCESSED_DATA_PATH / "4.6_test_df.csv", index=False)

In [40]:
X_test

,purchase_value,source,browser,sex,age,time_velocity,ip_user_share_count,device_user_share_count,day,hour,...,browser_FireFox,browser_IE,browser_Opera,browser_Safari,sex_F,sex_M,source_fr_enc,browser_fr_enc,source_targ_enc,browser_targ_enc
120889,12,SEO,FireFox,F,25,2805389.0,0,0,2,9,...,True,False,False,False,True,False,0.402071,0.162736,0.100379,0.107457
120890,16,SEO,IE,F,24,5228237.0,0,0,30,10,...,False,True,False,False,True,False,0.402071,0.242594,0.100379,0.097316
120891,75,SEO,Chrome,F,36,7858847.0,0,0,29,21,...,False,False,False,False,True,False,0.402071,0.406952,0.100379,0.111798
120892,29,Ads,IE,M,48,6591281.0,0,0,15,5,...,False,True,False,False,False,True,0.395388,0.242594,0.104356,0.097316
120893,26,Ads,IE,F,27,6171409.0,0,0,10,8,...,False,True,False,False,True,False,0.395388,0.242594,0.104356,0.097316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151107,16,SEO,FireFox,M,45,9101286.0,0,0,1,12,...,True,False,False,False,False,True,0.402071,0.162736,0.100379,0.107457
151108,34,SEO,Chrome,M,30,3578039.0,0,0,28,14,...,False,False,False,False,False,True,0.402071,0.406952,0.100379,0.111798
151109,22,Direct,Chrome,F,35,10295896.0,0,0,15,8,...,False,False,False,False,True,False,0.202541,0.406952,0.118195,0.111798
151110,33,Direct,IE,M,28,9574825.0,0,0,7,0,...,False,True,False,False,False,True,0.202541,0.242594,0.118195,0.097316


In [41]:
from collections import defaultdict

# Build a string report
report_lines = []
report_lines.append("DataFrame Summary")
report_lines.append("=" * 50)
report_lines.append(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
report_lines.append("\nSummary of Data Splitting Stage:")
report_lines.append(
"""
 Ordered based on signin time. Then split 80 to 20 splits.
"""
)
report_str = "\n".join(report_lines)

# Log to MLflow (within an active run)
mlflow.log_text(report_str, "ttsplit/summary.txt")

In [42]:
mlflow.end_run()

🏃 View run DataSplitting at: http://localhost:5000/#/experiments/1/runs/b8f6eb1e69dc42b296a8b951521defb2
🧪 View experiment at: http://localhost:5000/#/experiments/1
